# FAISS Document Ingestion and Index Creation

This notebook explains the full FAISS lifecycle:

```text
Documents → Embedding Model → NumPy float32 vectors → FAISS index → Search → Map results back to docs
```

Important:

> FAISS does not store text documents. FAISS stores vectors.  
> Your application keeps documents/metadata separately and maps FAISS result IDs back to them.

In [ ]:
# Install dependencies if needed:
# !pip install sentence-transformers faiss-cpu numpy pandas

In [ ]:
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

## 1. Create sample documents

These documents could come from PDFs, APIs, database rows, tickets, or knowledge articles.

In [ ]:
docs = [
    {
        "id": "doc-001",
        "title": "AWS Lambda",
        "category": "serverless",
        "text": "AWS Lambda is a serverless compute service that runs code without provisioning servers."
    },
    {
        "id": "doc-002",
        "title": "Amazon EC2",
        "category": "compute",
        "text": "Amazon EC2 provides resizable virtual machines for cloud compute workloads."
    },
    {
        "id": "doc-003",
        "title": "Amazon S3",
        "category": "storage",
        "text": "Amazon S3 is object storage built to store and retrieve any amount of data."
    },
    {
        "id": "doc-004",
        "title": "Azure Functions",
        "category": "serverless",
        "text": "Azure Functions is a serverless compute platform for event-driven applications."
    },
    {
        "id": "doc-005",
        "title": "Kubernetes",
        "category": "containers",
        "text": "Kubernetes orchestrates containerized applications across clusters."
    },
    {
        "id": "doc-006",
        "title": "Vector Search",
        "category": "search",
        "text": "Vector search uses embeddings and nearest neighbor algorithms to find semantically similar content."
    },
]

pd.DataFrame(docs)

## 2. Load an embedding model

We use `sentence-transformers/all-MiniLM-L6-v2`.

It converts text into a 384-dimensional vector.

In [ ]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

## 3. Convert documents into embeddings

This is the **document ingestion step**.

FAISS cannot ingest raw text. First, we convert text into vectors.

In [ ]:
texts = [doc["title"] + " " + doc["text"] for doc in docs]

embeddings = model.encode(
    texts,
    normalize_embeddings=True
).astype("float32")

print("Embeddings shape:", embeddings.shape)
print("Data type:", embeddings.dtype)

In [ ]:
# Preview first 10 dimensions of the first document vector
embeddings[0][:10]

## 4. Create a FAISS index

Because embeddings are normalized, we can use:

```python
faiss.IndexFlatIP(dim)
```

`IP` means **Inner Product**.

With normalized vectors:

```text
Inner Product ≈ Cosine Similarity
```

In [ ]:
dim = embeddings.shape[1]

index = faiss.IndexFlatIP(dim)

print("Index dimension:", dim)
print("Number of vectors before add:", index.ntotal)

## 5. Add vectors into FAISS

This is where the index is populated.

FAISS stores vectors in row order:

```text
row 0 → docs[0]
row 1 → docs[1]
row 2 → docs[2]
```

In [ ]:
index.add(embeddings)

print("Number of vectors after add:", index.ntotal)

## 6. Search the FAISS index

Query flow:

```text
user query → query embedding → FAISS search → returned row IDs → docs[row_id]
```

In [ ]:
query = "serverless compute without managing servers"

query_vec = model.encode(
    [query],
    normalize_embeddings=True
).astype("float32")

scores, ids = index.search(query_vec, k=3)

print("Scores:", scores)
print("FAISS row IDs:", ids)

## 7. Map FAISS result IDs back to documents

FAISS returns row positions, not document objects.

That is why this works:

```python
docs[i]
```

In [ ]:
results = []

for rank, row_id in enumerate(ids[0]):
    doc = docs[int(row_id)].copy()
    doc["rank"] = rank + 1
    doc["score"] = float(scores[0][rank])
    results.append(doc)

pd.DataFrame(results)[["rank", "id", "title", "category", "score", "text"]]

## 8. Better approach: use explicit numeric IDs

In production, you usually should not rely only on row position.

FAISS supports explicit numeric IDs using:

```python
faiss.IndexIDMap(...)
```

Important:

- FAISS IDs must be integers
- External document IDs may be strings
- Keep a mapping table between numeric FAISS IDs and your real document IDs

In [ ]:
faiss_id_to_doc_id = {
    101: "doc-001",
    102: "doc-002",
    103: "doc-003",
    104: "doc-004",
    105: "doc-005",
    106: "doc-006",
}

doc_id_to_doc = {doc["id"]: doc for doc in docs}

numeric_ids = np.array(list(faiss_id_to_doc_id.keys()), dtype="int64")

numeric_ids

In [ ]:
index_with_ids = faiss.IndexIDMap(faiss.IndexFlatIP(dim))

index_with_ids.add_with_ids(embeddings, numeric_ids)

print("Number of vectors:", index_with_ids.ntotal)

In [ ]:
scores2, faiss_ids = index_with_ids.search(query_vec, k=3)

print("Scores:", scores2)
print("Explicit FAISS IDs:", faiss_ids)

In [ ]:
explicit_results = []

for rank, faiss_id in enumerate(faiss_ids[0]):
    doc_id = faiss_id_to_doc_id[int(faiss_id)]
    doc = doc_id_to_doc[doc_id].copy()
    doc["rank"] = rank + 1
    doc["faiss_id"] = int(faiss_id)
    doc["score"] = float(scores2[0][rank])
    explicit_results.append(doc)

pd.DataFrame(explicit_results)[["rank", "faiss_id", "id", "title", "category", "score", "text"]]

## 9. Save and load FAISS index

FAISS indexes can be saved to disk.

Important:

- FAISS saves the vector index
- FAISS does not save your Python `docs` list
- You must separately persist document metadata

In [ ]:
faiss.write_index(index_with_ids, "faiss_index_with_ids.index")

print("Saved FAISS index.")

In [ ]:
loaded_index = faiss.read_index("faiss_index_with_ids.index")

scores3, faiss_ids3 = loaded_index.search(query_vec, k=3)

print("Scores:", scores3)
print("IDs:", faiss_ids3)

## 10. Complete helper class

This wraps the full lifecycle:

- load docs
- generate embeddings
- create FAISS index
- add vectors
- search
- map results back to documents

In [ ]:
class FaissDocumentSearch:
    def __init__(self, docs, model_name="sentence-transformers/all-MiniLM-L6-v2"):
        self.docs = docs
        self.model = SentenceTransformer(model_name)

        self.doc_id_to_doc = {doc["id"]: doc for doc in docs}

        self.faiss_id_to_doc_id = {
            i: doc["id"]
            for i, doc in enumerate(docs)
        }

        texts = [
            doc["title"] + " " + doc["text"]
            for doc in docs
        ]

        self.embeddings = self.model.encode(
            texts,
            normalize_embeddings=True
        ).astype("float32")

        dim = self.embeddings.shape[1]

        self.index = faiss.IndexIDMap(faiss.IndexFlatIP(dim))

        numeric_ids = np.array(
            list(self.faiss_id_to_doc_id.keys()),
            dtype="int64"
        )

        self.index.add_with_ids(self.embeddings, numeric_ids)

    def search(self, query, k=3):
        query_vec = self.model.encode(
            [query],
            normalize_embeddings=True
        ).astype("float32")

        scores, faiss_ids = self.index.search(query_vec, k)

        results = []
        for rank, faiss_id in enumerate(faiss_ids[0]):
            doc_id = self.faiss_id_to_doc_id[int(faiss_id)]
            doc = self.doc_id_to_doc[doc_id].copy()
            doc["rank"] = rank + 1
            doc["score"] = float(scores[0][rank])
            doc["faiss_id"] = int(faiss_id)
            results.append(doc)

        return results

In [ ]:
engine = FaissDocumentSearch(docs)

final_results = engine.search("cloud object storage", k=3)

pd.DataFrame(final_results)[["rank", "faiss_id", "id", "title", "category", "score", "text"]]

## 11. Final mental model

```text
FAISS stores:
  vector[0]
  vector[1]
  vector[2]

Your app stores:
  docs[0]
  docs[1]
  docs[2]

FAISS search returns:
  [2, 0, 1]

Your app maps:
  docs[2], docs[0], docs[1]
```

Production systems usually store metadata in:

- PostgreSQL
- OpenSearch
- DynamoDB
- S3 JSON files
- document databases

FAISS is the vector search engine, not the document database.